In [1]:
from google.cloud import bigquery
client = bigquery.Client()

query = """ SELECT concat(title,'. ',text) as text FROM `ingka-feed-student-dev.RR.RatingsReviews` AS rr
            INNER JOIN `ingka-feed-student-dev.RR.product_categories` AS pc 
            ON rr.art_id = SPLIT(pc.global_id, ',')[SAFE_OFFSET(1)] 
            WHERE country_code = 'us' and PRODUCT_AREA = 'Open storage'
            ORDER BY inserted_on DESC """

query_job = client.query(query)

reviews = []

for review in query_job:
     reviews.append(review.text)

reviews[0]

'Extra shelves. These extra shelves make waisted space not waisted. '

In [2]:
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /var/tmp/pip-req-build-75o4jxr2
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /var/tmp/pip-req-build-75o4jxr2
  Resolved https://github.com/huggingface/transformers to commit af4c02622bfb4521367c459c6743014ef9be788d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!pip install accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_model_id = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,  # Mistral, same as before
    quantization_config=bnb_config,  # Same quantization config as before
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    add_bos_token=True,
    trust_remote_code=True,
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
from peft import PeftModel

model = PeftModel.from_pretrained(base_model, "master-thesis/Experiments/results/mistral7b-instructv02-finetune-400-openstorage/checkpoint-1000")

ValueError: Can't find 'adapter_config.json' at 'master-thesis/Experiments/results/mistral7b-instructv02-finetune-400-openstorage/checkpoint-1000/'

In [ ]:
import time

start_time = time.time()
device = "cuda"
answers = []
pre_text = 0
eos_token_id = tokenizer.convert_tokens_to_ids('.')
eos_token_id_additional = tokenizer.convert_tokens_to_ids('<')

for review in reviews: 
    review_len = len(review)
    
    messages = [
        {"role": "user", "content": "From this sentence: 'Very sturdy and sleek. Super easy to assemble, sturdy materials and construction.  I love how thin looking the shelves are: makes all your items really pop out.  Being able to choose the shelves spacing is a plus.  Sturdy enough for large books.' generate a keyphrase for each key characteristic of the product and classify the sentiment of each generated keyphrase between positive, negative or neutral. Don't give any explainations, the output should only be Keyphrase 1 - Sentiment 1, Keyphrase 2 - Sentiment 2, ..."},
        {"role": "assistant", "content": "Sturdy - positive, sleek - positive, easy to assemble - positive"},
        {"role": "user", "content": f"From this sentence: '{review}' generate a keyphrase for each key characteristic of the product and classify the sentiment of each generated keyphrase between positive, negative or neutral. Don't give any explainations, the output should only be Keyphrase 1 - Sentiment 1, Keyphrase 2 - Sentiment 2, ..."},
    ]
    
    
    encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")

    model_inputs = encodeds.to(device)

    generated_ids = model.generate(model_inputs, max_new_tokens=75, pad_token_id=eos_token_id, eos_token_id=eos_token_id, do_sample=True)
    decoded = tokenizer.batch_decode(generated_ids)
    
    
    answer = decoded[0]
    
    if eos_token_id_additional != -1:
        answer = answer.split('<')[0]
        
    answers.append(answer)
    
    if (answers.index(answer) + 1) % 100 == 0: 
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"{answers.index(answer) + 1}th file completed in {elapsed_time} seconds.")

In [ ]:
reviews[19]

In [ ]:
reviews = [review.replace('\n', ' ') for review in reviews]

In [ ]:
reviews[19]

In [ ]:
answers[19]

In [ ]:
import csv

filename = 'New_Reviews_keyphrases_400_US.csv'

# Writing to the csv file
with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
    # Creating a csv writer object with quoting set to quote all fields
    csvwriter = csv.writer(csvfile, delimiter=';')
    
    # Writing the columns
    csvwriter.writerow(['Reviews', 'Keyphrases'])
    
    # Writing the data
    for review, answer in zip(reviews, answers):
        csvwriter.writerow([review, answer])

print(f'CSV file "{filename}" has been written successfully.')